# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

# Filter out documents with poor content quality and structure them properly
filtered_docs = []
for doc in loan_complaint_data:
    narrative = doc.metadata.get("Consumer complaint narrative", "")
    
    # Skip documents with insufficient content or too many redactions
    if (len(narrative.strip()) < 100 or 
        narrative.count("XXXX") > 5 or 
        narrative.strip() in ["", "None", "N/A"]):
        continue
    
    # Create meaningful page_content by combining narrative with context
    doc.page_content = f"Customer Issue: {doc.metadata.get('Issue', 'Unknown')}\n"
    doc.page_content += f"Product: {doc.metadata.get('Product', 'Unknown')}\n"
    doc.page_content += f"Complaint Details: {narrative}"
    
    filtered_docs.append(doc)

# Use filtered documents instead
loan_complaint_data = filtered_docs[:20]  # Start with smaller subset


Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, the most common issue with loans appears to be problems related to "Dealing with your lender or servicer," specifically issues such as trouble with how payments are being handled, receiving bad information about the loan, unauthorized access to personal information, and discrepancies or errors in loan reporting and status. Many complaints involve mismanagement of payments, interest accrual issues, lack of communication, and security breaches.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, several complaints indicate issues with timely handling or resolution. Specifically, while some responses to complaints were marked as "Yes" for being timely, others involve unresolved or ongoing issues, such as disconnections, incorrect information, or lack of follow-up. Furthermore, most complaints received a response of "Closed with explanation," suggesting they were addressed, but often not to the complainants\' satisfaction or with unresolved concerns.\n\nTherefore, it appears that some complaints did not get handled in a timely manner, or at least not to the full satisfaction of the complainants.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with how payments were being handled or re-amortized after forbearance periods, problems with autopay setup or processing, receiving incorrect or bad information about their loan status, and administrative errors such as loans being wrongly reported as in default or delinquent. Additionally, some faced complications due to technical issues, lack of communication, or presumed unauthorized access and data breaches affecting their personal information.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with student loan complaints appears to be problems related to "Dealing with your lender or servicer," particularly trouble with how payments are being handled. Multiple complaints mention issues such as re-amortization after forbearance, management of automatic payments, incorrect payment statuses, and difficulties in communication with the servicers.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints listed indicate that they were responded to in a timely manner. For example, the complaint from April 14, 2025, was responded to on April 15, 2025, and the complaint from April 28, 2025, was responded to on the same day, both with responses marked as "Yes" for timely response. Therefore, there is no evidence in this data to suggest that any complaints were not handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'Based on the provided complaints, people failed to pay back their loans primarily because of issues related to the handling and servicing of their student loans. These issues include autopay setup problems that prevented automatic payments from processing, miscommunications or lack of notifications about missed payments, and errors or mismanagement by loan servicers. Additionally, some borrowers experienced data breaches, violations of privacy laws, and incorrect account statuses, which negatively impacted their credit reports and financial situation. These administrative and technical problems made it difficult for borrowers to make or track their payments, leading to missed payments and, in some cases, default.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
####✅ Answer: What was the weather in Charlotte yesterday? Because BM25 works better when the query need retrival based on precise keyword matches  vs abstract and semantic generalization needs

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issue with student loans appears to be problems related to dealing with lenders or servicers, specifically issues with how payments are being handled. Many complaints mention trouble with achieving correct payment processing, auto-pay reversals, and communication problems with loan servicers.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that all of them were responded to in a timely manner, as each has a note indicating "Timely response?": "Yes." Therefore, there is no evidence in this data that any complaints were not handled in a timely manner.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including problems with how their payments were being handled, issues with autopay setups, miscommunication or lack of notification about payment statuses, and errors or incorrect information regarding their account status. In some cases, personal data breaches or system errors contributed to difficulties in repayment.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issue with loans appears to be problems related to dealing with lenders or servicers, particularly issues with payments, miscommunication, incorrect loan information, or mishandling of data. Many complaints involve trouble with payment handling, incorrect information on credit reports, delays or failures in communication, and concerns about data privacy violations.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that they were handled in a timely manner. For example, complaints with responses marked as "Closed with explanation" also note "Timely response? Yes," and responses were provided within an appropriate timeframe. However, there is no explicit mention of complaints that were not handled in a timely manner. \n\nTherefore, I do not have evidence from this data to confirm that any complaints were not handled promptly.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues such as encountering inaccurate or unjustified account status reports, lack of communication from lenders or servicers, problems with how payments were being handled (e.g., interest accruing despite promises of freezing, automatic payment reversals, or inability to make payments), and complications related to data breaches or unauthorized access to personal information. Additionally, some individuals faced difficulties due to misunderstandings about their loan status or legal complications arising from the abolishment of certain education agencies, which affected the legitimacy and verification of their debts. Overall, these issues often stem from mismanagement, lack of transparency, or errors by loan servicers, leading to defaults or payment problems.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
###✅ Answer: I can't expect the user to be trained on prompting, and my app should handle queries written in all type of prose and loaded with errors. query reformulation would allow the retriver to retrive relevant information that semantically matches several reforumulations of the original query (ideally matching the actul intent of the query better). It increases retrival diversity, by retriving context that may come from lexically variant sources.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to dealing with lenders or servicers, specifically issues with how payments are handled, including autopay failures, repeated reversals, and difficulties with account management and communication. Many complaints highlight trouble with payment processing, lack of transparency, and poor communication from loan servicers.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that they were not handled in a timely manner. Specifically, the complaint from the individual who set up autopay and received a past-due notice after 20 days, despite having confirmation that autopay was set up, suggests there was a delay or issue in processing or communicating the payment setup. Additionally, the complaint about difficulty reaching customer service with long wait times and disconnections points to issues in timely support. \n\nHowever, the responses to all these complaints are marked as "Closed with explanation," and the responses were marked as "Timely response: Yes." This indicates that, from the company\'s perspective, responses were handled within the expected timeframe, though the consumer still experienced delays or issues.\n\nIn summary, while the company responded promptly in terms of timing, some complaints reflect that issues were not resolved as swiftly as needed for the consumer\'s satisfact

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons as evidenced by the complaints. Some common reasons include:\n\n1. Legal and administrative issues, such as loans being reported illegally or without verification, especially following the abolishment of certain agencies like the Department of Education.\n2. Confusion or lack of clear communication about the status of loans, payments, and repayment terms, leading to misunderstandings or mismanagement.\n3. Data breaches and mishandling of personal information, which can impact a borrower’s ability or willingness to continue repayment.\n4. Financial hardship or changed financial circumstances, making it difficult for borrowers to meet increased payments or manage their loan obligations.\n5. Dissatisfaction with the service or handling of their loans, including poor communication, unverified debts, or disputes over the legitimacy of debt.\n\nOverall, a combination of legal, administrative, communication, privacy, and financial iss

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be problems related to "Dealing with your lender or servicer," especially issues such as trouble with how payments are being handled, incorrect information about the loan, and data breaches or mishandling of personal information. Many complaints also involve poor communication, unauthorized data sharing, and technical issues with account management.'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints mention delays or issues with handling issues in a timely manner. For example:\n\n- One complaint from 04/28/25 describes a delay of about 20 days in setting up autopay, with the complainant receiving no notification that the autopay was incomplete.\n- Multiple complaints mention that responses from companies were "Closed with explanation," but they also note issues such as disconnections, lack of follow-up, or insufficient communication, suggesting possible delays or lapses in handling.\n- Specifically, complaints with issues like trouble with payments, incorrect information, or data breaches do not explicitly state they were not handled on time, but the nature of the complaints and the responses imply some level of delay or mishandling.\n\nIn summary, yes, some complaints indicate that issues were not handled in a timely manner, with delays ranging from several days to weeks.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including:\n\n1. **Incorrect or confusing information regarding their loan status**, such as being notified of default without prior indication or never having been in default, leading to dropped credit scores and financial hardship.\n\n2. **Problems with loan servicing and handling**, such as delays or failures in processing payments, incorrect billing amounts, or issues with autopay setups, which can cause missed payments or account delinquencies.\n\n3. **Data breaches and mishandling of personal information**, leading to breaches of privacy laws (like FERPA, Privacy Act, and Higher Education Act), which can undermine trust and cause disruptions in loan management.\n\n4. **Disputes over loan legitimacy or legal status**, such as believing certain loans or collections are illegitimate or invalid, sometimes due to legislative changes or executive orders.\n\n5. **Financial hardships or misunderstandings about repayment obligati

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [43]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to dealing with lenders or servicers. This includes issues such as trouble with how payments are being handled, receiving incorrect information about the loan, disputes over payment records, and difficulties in communication or obtaining clear information. Many complaints also involve concerns about data breaches or mishandling of personal information, but the recurring theme across multiple complaints is the difficulty borrowers face when interacting with their loan servicers or lenders.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, all of them indicate that responses from the companies were marked as "Closed with explanation" and responses were provided in a timely manner. There is no evidence in the data suggesting that any complaints were not handled within the expected timeframes.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with loan servicing, mismanagement, lack of transparency, and data breaches. Specific issues cited include payments not being properly re-amortized after forbearance periods ended, problems with auto pay reversals and incorrect account information, undue stress caused by poor communication and long wait times when seeking assistance, and disputes over interest charges and loan status due to alleged violations or mismanagement by servicers. Additionally, some borrowers experienced unauthorized data sharing and breaches of privacy, which further complicated their ability to manage repayment.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
### ✅✅✅ Answer: If questions are repetitive, semantic chunking will merge the questions into similar chunks, losing distinction of questions, and providing redundant, blended, generic answers. One way is to not use semantic chunking for questions in FAQ, but alternatively, reduce the chunk size with clear seperators to presever the units within each FAQ. Second option is to treat each questions as a separate chunk. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [49]:
#NLTK Import To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/chrag/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/chrag/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [50]:
### Step 0: Dependencies & imports

# included in pyproject.toml file
# "numpy>=2.2.2",
# "ragas==0.3.0",
# "rapidfuzz"
# "langchain-core",
# "langchain-community",
# "langsmith",
#  "tqdm", 


#Setting up the LLM and embedding model and generator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


# Use a more capable model for complex knowledge extraction tasks
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",  # More capable than nano for reasoning tasks
    temperature=0.1,      # Lower temperature for more consistent outputs
    request_timeout=120   # Longer timeout for complex operations
))

generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [64]:
#Step 1 Create a "golden dataset" a.k.a synthetic test data
# This will generate our knowledge graph under the hood and generate our personas and scenarios to construct our queries

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
sample_docs = loan_complaint_data[:15]  # subset of full data set

from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]


try:
    print(f"Generating test dataset with {len(sample_docs)} documents...")
    dataset = generator.generate_with_langchain_docs(
        sample_docs,
        testset_size=10,
        query_distribution=query_distribution,  # adding query distribution to the generator
    )
    print("Dataset generation completed successfully!")
    print(f"Generated {len(dataset)} test samples")
except Exception as e:
    print(f"Error during dataset generation: {e}")
    print("Try reducing document count or testset_size further")

Generating test dataset with 15 documents...


Applying SummaryExtractor:   0%|          | 0/7 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/15 [00:00<?, ?it/s]

Node 4071b891-174c-481c-a7a2-875a260df80a does not have a summary. Skipping filtering.
Node 2ce51cb5-8a28-45f5-9f53-75b991c57e8a does not have a summary. Skipping filtering.
Node 19c815a2-5adc-40eb-a6a0-7abc6cf0e7f4 does not have a summary. Skipping filtering.
Node 8cf7f506-0f03-4a52-b610-03da2206e559 does not have a summary. Skipping filtering.
Node b4331093-ff9f-4f4e-903d-234bd3a979b8 does not have a summary. Skipping filtering.
Node b23fba1d-93f6-4bf1-9989-2e8937d7a7d4 does not have a summary. Skipping filtering.
Node d4cb3b63-af2b-465d-af1e-1e1c17f98b97 does not have a summary. Skipping filtering.
Node 35913f0c-4ed9-4566-95ec-be3ee4b4801b does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/37 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

Dataset generation completed successfully!
Generated 11 test samples


In [65]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Why payments go up after COVID-19 forbearance ...,[Customer Issue: Dealing with your lender or s...,The payments increased because the federal stu...,single_hop_specifc_query_synthesizer
1,What issues can arise with IDR applications ba...,[Customer Issue: Dealing with your lender or s...,The customer experienced issues with Aidvantag...,single_hop_specifc_query_synthesizer
2,What happen when FERPA is violated in student ...,[Customer Issue: Dealing with your lender or s...,"When FERPA is violated in a student loan case,...",single_hop_specifc_query_synthesizer
3,What issues is the borrower facing with Nelnet...,[Customer Issue: Dealing with your lender or s...,The borrower is experiencing confusion regardi...,single_hop_specifc_query_synthesizer
4,Why is it so hard to get help with my student ...,[Customer Issue: Dealing with your lender or s...,Dealing with your lender or servicer for your ...,single_hop_specifc_query_synthesizer
5,What are the complaint details regarding custo...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Borrowers dealing with Nelnet as their student...,multi_hop_abstract_query_synthesizer
6,What issues did borrowers face regarding autop...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Borrowers faced significant issues related to ...,multi_hop_abstract_query_synthesizer
7,What are the common customer complaints regard...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Common customer complaints regarding student l...,multi_hop_abstract_query_synthesizer
8,What are the implications of FERPA violations ...,[<1-hop>\n\nCustomer Issue: Improper use of yo...,The implications of FERPA violations in relati...,multi_hop_specific_query_synthesizer
9,What issues are borrowers facing with their lo...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Borrowers are facing issues such as incomplete...,multi_hop_specific_query_synthesizer


In [66]:
#all imports
import copy
import time
import pandas as pd
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from langchain_core.runnables import Runnable

In [67]:
#setting up LangSmith api and project   
import os
import getpass
from langsmith import Client
from langsmith import traceable

os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("🔐 LangSmith API Key: ") # Securely input your LangSmith API key
os.environ["LANGCHAIN_PROJECT"] = "09_Advanced_Retrieval" # Set project name (must be via environment var)
os.environ["LANGCHAIN_TRACING_V2"] = "true" # enables LangSmith's latest tracing system, known as "Tracing V2"

client = Client() # Initialize LangSmith client

In [69]:
#defines evaluate function for retriever metrics

@traceable(run_type="evaluation", name="Evaluate Retriever", tags=["retriever"])
def evaluate_retriever(
    dataset,
    retriever_chain: Runnable,
    delay_sec: float = 1.0,
    model_name: str = "gpt-4.1-mini",
    timeout: int = 360,
    verbose: bool = False,
    return_dataset: bool = False,
):
    eval_dataset = copy.deepcopy(dataset)

    for i, test_row in enumerate(eval_dataset):
        user_question = test_row.eval_sample.user_input

        if verbose:
            print(f"[{i+1}/{len(eval_dataset)}] Querying retriever: {user_question}")

        result = retriever_chain.invoke({"question": user_question})

        test_row.eval_sample.retrieved_contexts = [
            doc.page_content for doc in result["context"]
        ]
        test_row.eval_sample.response = " "

        time.sleep(delay_sec)

    df = pd.DataFrame([row.eval_sample.to_dict() for row in eval_dataset])
    df["response"] = df["response"].fillna(" ")
    ragas_dataset = EvaluationDataset.from_pandas(df)

    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model=model_name))
    run_config = RunConfig(timeout=timeout)

    retriever_metrics = [
        LLMContextRecall(),        # Measures how much of the relevant context (needed to answer the query) is retrieved
        ContextEntityRecall(),     #  Measures whether key entities from the gold/reference answer are present in the retrieved context.
        ContextPrecision(),        #  easures the proportion of relevant chunks in the retrieved contexts
        NoiseSensitivity(),         #  easures how often a system makes errors by providing incorrect responses
    ]

    results = evaluate(
        dataset=ragas_dataset,
        metrics=retriever_metrics,
        llm=evaluator_llm,
        run_config=run_config,
    )

    return (results, eval_dataset) if return_dataset else results

In [70]:
 # Copy dataset to avoid modifying original
eval_dataset = copy.deepcopy(dataset)

In [71]:
# Call the evaluator function for naive retriever
naive_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=naive_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(naive_results)

[1/11] Querying retriever: Why payments go up after COVID-19 forbearance end?
[2/11] Querying retriever: What issues can arise with IDR applications based on the customer's experience?
[3/11] Querying retriever: What happen when FERPA is violated in student loan case?
[4/11] Querying retriever: What issues is the borrower facing with Nelnet as their loan servicer?
[5/11] Querying retriever: Why is it so hard to get help with my student loan and what can I do about it?
[6/11] Querying retriever: What are the complaint details regarding customer issues faced by borrowers dealing with Nelnet as their student loan servicer?
[7/11] Querying retriever: What issues did borrowers face regarding autopay setup and payment increases after the COVID-19 forbearance program ended?
[8/11] Querying retriever: What are the common customer complaints regarding student loan servicers, particularly in relation to payment amounts and processing issues, as highlighted in the complaints submitted to Aidvanta

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.9773, 'context_entity_recall': 0.4880, 'context_precision': 0.6872, 'noise_sensitivity(mode=relevant)': nan}


### Naive retriver {'context_recall': 0.9773, 'context_entity_recall': 0.4880, 'context_precision': 0.6872, 'noise_sensitivity(mode=relevant)': nan}

In [72]:
# Call the evaluator function  for bm25
bm25_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=bm25_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(bm25_results)

[1/11] Querying retriever: Why payments go up after COVID-19 forbearance end?
[2/11] Querying retriever: What issues can arise with IDR applications based on the customer's experience?
[3/11] Querying retriever: What happen when FERPA is violated in student loan case?
[4/11] Querying retriever: What issues is the borrower facing with Nelnet as their loan servicer?
[5/11] Querying retriever: Why is it so hard to get help with my student loan and what can I do about it?
[6/11] Querying retriever: What are the complaint details regarding customer issues faced by borrowers dealing with Nelnet as their student loan servicer?
[7/11] Querying retriever: What issues did borrowers face regarding autopay setup and payment increases after the COVID-19 forbearance program ended?
[8/11] Querying retriever: What are the common customer complaints regarding student loan servicers, particularly in relation to payment amounts and processing issues, as highlighted in the complaints submitted to Aidvanta

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.8924, 'context_entity_recall': 0.3765, 'context_precision': 0.6061, 'noise_sensitivity(mode=relevant)': nan}


### BM25 retriver {'context_recall': 0.8924, 'context_entity_recall': 0.3765, 'context_precision': 0.6061, 'noise_sensitivity(mode=relevant)': nan}

In [76]:
# Call the evaluator function  for Cohere Reranker
CohereReranker_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=contextual_compression_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True,
    delay_sec=7.0 #wait 7 seconds between calls (10 calls/min = 6 sec intervals + buffer)
)

print(CohereReranker_results)

[1/11] Querying retriever: Why payments go up after COVID-19 forbearance end?
[2/11] Querying retriever: What issues can arise with IDR applications based on the customer's experience?
[3/11] Querying retriever: What happen when FERPA is violated in student loan case?
[4/11] Querying retriever: What issues is the borrower facing with Nelnet as their loan servicer?
[5/11] Querying retriever: Why is it so hard to get help with my student loan and what can I do about it?
[6/11] Querying retriever: What are the complaint details regarding customer issues faced by borrowers dealing with Nelnet as their student loan servicer?
[7/11] Querying retriever: What issues did borrowers face regarding autopay setup and payment increases after the COVID-19 forbearance program ended?
[8/11] Querying retriever: What are the common customer complaints regarding student loan servicers, particularly in relation to payment amounts and processing issues, as highlighted in the complaints submitted to Aidvanta

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.9288, 'context_entity_recall': 0.4636, 'context_precision': 0.6818, 'noise_sensitivity(mode=relevant)': nan}


### Cohere Reranker {'context_recall': 0.9288, 'context_entity_recall': 0.4636, 'context_precision': 0.6818, 'noise_sensitivity(mode=relevant)': nan}

In [77]:
# Call the evaluator function  for MultiQueryRetriever
MultiQueryRetriever_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=multi_query_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(MultiQueryRetriever_results)

[1/11] Querying retriever: Why payments go up after COVID-19 forbearance end?
[2/11] Querying retriever: What issues can arise with IDR applications based on the customer's experience?
[3/11] Querying retriever: What happen when FERPA is violated in student loan case?
[4/11] Querying retriever: What issues is the borrower facing with Nelnet as their loan servicer?
[5/11] Querying retriever: Why is it so hard to get help with my student loan and what can I do about it?
[6/11] Querying retriever: What are the complaint details regarding customer issues faced by borrowers dealing with Nelnet as their student loan servicer?
[7/11] Querying retriever: What issues did borrowers face regarding autopay setup and payment increases after the COVID-19 forbearance program ended?
[8/11] Querying retriever: What are the common customer complaints regarding student loan servicers, particularly in relation to payment amounts and processing issues, as highlighted in the complaints submitted to Aidvanta

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.9773, 'context_entity_recall': 0.4346, 'context_precision': 0.6521, 'noise_sensitivity(mode=relevant)': nan}


### MultiQueryRetriever_results {'context_recall': 0.9773, 'context_entity_recall': 0.4346, 'context_precision': 0.6521, 'noise_sensitivity(mode=relevant)': nan}

In [78]:
# Call the evaluator function  for Parent Document Retriever
parent_document_retriever_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=parent_document_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(parent_document_retriever_results)

[1/11] Querying retriever: Why payments go up after COVID-19 forbearance end?
[2/11] Querying retriever: What issues can arise with IDR applications based on the customer's experience?
[3/11] Querying retriever: What happen when FERPA is violated in student loan case?
[4/11] Querying retriever: What issues is the borrower facing with Nelnet as their loan servicer?
[5/11] Querying retriever: Why is it so hard to get help with my student loan and what can I do about it?
[6/11] Querying retriever: What are the complaint details regarding customer issues faced by borrowers dealing with Nelnet as their student loan servicer?
[7/11] Querying retriever: What issues did borrowers face regarding autopay setup and payment increases after the COVID-19 forbearance program ended?
[8/11] Querying retriever: What are the common customer complaints regarding student loan servicers, particularly in relation to payment amounts and processing issues, as highlighted in the complaints submitted to Aidvanta

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.7985, 'context_entity_recall': 0.3846, 'context_precision': 0.7121, 'noise_sensitivity(mode=relevant)': nan}


### parent_document_retriever_results {'context_recall': 0.7985, 'context_entity_recall': 0.3846, 'context_precision': 0.7121, 'noise_sensitivity(mode=relevant)': nan}

In [79]:
# Call the evaluator function  for Ensemble Retriever
ensemble_retrieval_chain_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=ensemble_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(ensemble_retrieval_chain_results)

[1/11] Querying retriever: Why payments go up after COVID-19 forbearance end?
[2/11] Querying retriever: What issues can arise with IDR applications based on the customer's experience?
[3/11] Querying retriever: What happen when FERPA is violated in student loan case?
[4/11] Querying retriever: What issues is the borrower facing with Nelnet as their loan servicer?
[5/11] Querying retriever: Why is it so hard to get help with my student loan and what can I do about it?
[6/11] Querying retriever: What are the complaint details regarding customer issues faced by borrowers dealing with Nelnet as their student loan servicer?
[7/11] Querying retriever: What issues did borrowers face regarding autopay setup and payment increases after the COVID-19 forbearance program ended?
[8/11] Querying retriever: What are the common customer complaints regarding student loan servicers, particularly in relation to payment amounts and processing issues, as highlighted in the complaints submitted to Aidvanta

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.9773, 'context_entity_recall': 0.3925, 'context_precision': 0.7058, 'noise_sensitivity(mode=relevant)': nan}


### ensemble_retrieval {'context_recall': 0.9773, 'context_entity_recall': 0.3925, 'context_precision': 0.7058, 'noise_sensitivity(mode=relevant)': nan}

## ✅✅✅ ANALYSIS 

| Retriever                  | Context Recall | Entity Recall | Context Precision |
| -------------------------- | -------------- | ------------- | ----------------- |
| **Ensemble**               | **0.9773**     | 0.3925        | **0.7058**        |
| Parent Document            | 0.7985         | 0.3846        | 0.7121            |
| Multi-query                | **0.9773**     | 0.4346        | 0.6521            |
| **Contextual Compression** | 0.9288         | 0.4636        | 0.6818            |
| BM25                       | 0.8924         | 0.3765        | 0.6061            |
| Naive                      | **0.9773**     | **0.4880**    | 0.6872            |


Ensemble achieves the highest precision (0.7058) among retrievers with high recall (0.9773).

Contextual Compression has slightly lower recall and precision but highest entity recall after Naive.

Naive retriever technically has the best entity recall, but its lack of nuance makes it less scalable for real-world use cases.

Multi-query is high-recall but lower in precision.

Combining Ensemble + Cohere Reranker would likely boost both coverage and focus, while minimizing trade-offs—even though not explicitly tested together.